In [ ]:
import sys
sys.path.append('../../')

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.gridspec import GridSpec

from gaussianproc import CurveStats, ImageStats
fw_func = CurveStats.full_width
gaussian1d_func = CurveStats.gaussian

# load file

In [ ]:
filedir = "/home/ids/data/2025-07-16-Slits_check"
filename = 'square_scan_20250716_01.h5'
file = '/'.join([filedir,filename])
f = h5py.File(name=file, mode='r')

scanames = list(f.keys())

str_func = lambda s: [int(x) for x in s.split('-')[1:]]
scanames = sorted(scanames, key=str_func)

# check energy

In [ ]:
energies = []

for scan in scanames:
    dvf2name = '/'.join([scan,'dvf2-img'])
    arr2 = np.array(f[dvf2name])
    energies.append(np.sum(arr2))

In [ ]:
plt.plot(energies)
plt.xlabel('scan idx')
plt.ylabel('energy')

plt.hlines(1.4e6,0,160,colors='k')

plt.show()

# fwhm analysis

store widths

In [ ]:

fwhmsx = []
fwhmsy = []


for scan in scanames:
    print(scan)
    
    dvf2name = '/'.join([scan,'dvf2-img'])
    arr = np.array(f[dvf2name])
    CurveStats.ECUTOFF = 1_400_000
    ImageStats.ECUTOFF = 1_400_000
    image = ImageStats(data=arr)

    if not image.is_empty():

        factor = 2 * np.sqrt(2*np.log(2))

        sigx = image.projx.gaussianfitparams['sigma']
        fwhmx = 0.48 * factor * sigx
        sigy = image.projy.gaussianfitparams['sigma']
        fwhmy = 0.48 * factor * sigy

        fwhmsx.append(fwhmx)
        fwhmsy.append(fwhmy)

    else:

        fwhmsx.append(np.nan)
        fwhmsy.append(np.nan)

        # image.plot()

fwhmsx = np.array(fwhmsx).reshape(-1,7)
fwhmsy = np.array(fwhmsy).reshape(-1,7)

optimization

In [ ]:
idxminxflat = np.nanargmin(fwhmsx)
idxminyflat = np.nanargmin(fwhmsy)
idxminx = np.unravel_index(idxminxflat, fwhmsx.shape)
idxminy = np.unravel_index(idxminyflat, fwhmsy.shape)

In [ ]:
# matrices of fwhms #
print(fwhmsx.round(3))
print()
print(fwhmsy.round(3))

In [ ]:
# fwhm min #

# from found min fwhmx
print(f'fwhmx: {fwhmsx[idxminx]} um\nfwhmy: {fwhmsy[idxminx]} um')

print()

# from found min fwhmy
print(f'fwhmx: {fwhmsx[idxminy]} um\nfwhmy: {fwhmsy[idxminy]} um')

In [ ]:
plt.subplot(1,2,1)
plt.title('FWHMx')
plt.imshow(fwhmsx,origin='lower')
plt.plot(idxminx[1],idxminx[0],'ro')
plt.colorbar(label='width [um]')

plt.subplot(1,2,2)
plt.title('FWHMy')
plt.imshow(fwhmsy,origin='lower')
plt.plot(idxminy[1],idxminy[0],'ro')
plt.colorbar(label='width [um]')

plt.tight_layout()
plt.show()

In [ ]:
# slit positions for found min fwhmx #
dict(f[scanames[idxminxflat]].attrs)

In [ ]:
# slit positions for found min fwhmy #
dict(f[scanames[idxminyflat]].attrs)